# TypedDict

In [1]:
from langchain_openai import ChatOpenAI
from typing import List, Optional, Annotated, TypedDict
from dotenv import load_dotenv
import os

# Load .env
load_dotenv()

# Create LLM
llm = ChatOpenAI(
    model="deepseek-ai/deepseek-v4-pro",
    temperature=0.6,
    api_key=os.getenv("DEEPSEEK_API_KEY_PRO"),
    base_url="https://integrate.api.nvidia.com/v1"
)

# Define structured output schema
class Movie(TypedDict):

    title: Annotated[str, "Name of the movie"]
    release_year: Annotated[int, "Year the movie was released"]
    genres: Annotated[List[str], "List of genres"]
    rating: Annotated[float, "Movie rating from 1-10"]
    box_office: Annotated[
        Optional[float],
        "Worldwide box office in millions USD"
    ]

# Convert LLM to structured LLM output
structured_llm = llm.with_structured_output(Movie)

# Invoke model
result = structured_llm.invoke(
    "Give me details of a movie named Inception"
)

# Print result
print(result)
print("\n\n")

# Access values
print(result["title"])
print(result["rating"])

{'title': 'Inception', 'release_year': 2010, 'genres': ['Action', 'Sci-Fi', 'Thriller'], 'rating': 8.8, 'box_office': 836.8}



Inception
8.8


# Pydantic

In [ ]:
from langchain_openai import ChatOpenAI
from pydantic import BaseModel, Field
from typing import List, Optional
from dotenv import load_dotenv
import os

# Load .env
load_dotenv()

# Create LLM
llm = ChatOpenAI(
    model="deepseek-ai/deepseek-v4-pro",
    temperature=0.6,
    api_key=os.getenv("DEEPSEEK_API_KEY_PRO"),
    base_url="https://integrate.api.nvidia.com/v1"
)

# Create schema using Pydantic
class Movie(BaseModel):

    # ... - required field
    title: str = Field(..., description="Name of the movie")
    release_year: int = Field(...,description="Year the movie was released")
    genres: List[str] = Field(..., description="List of genres")
    rating: float = Field(..., description="Movie rating from 1 to 10")
    box_office: Optional[float] = Field(...,description="Worldwide box office in millions USD")

# Structured LLM
structured_llm = llm.with_structured_output(Movie)

# Invoke model
result = structured_llm.invoke(
    "Give me details about the movie Inception."
)

print(result)

print(result.model_dump())
print(result.model_dump_json())

movie = Movie(
    genres=["Sci-Fi", "Action", "Adventure"],
    title="Inception",
    release_year=2010,
    rating=8.8,
    box_office=None
)

movie

title='Inception' release_year=2010 genres=['Action', 'Science Fiction', 'Thriller'] rating=8.8 box_office=836.8
{'title': 'Inception', 'release_year': 2010, 'genres': ['Action', 'Science Fiction', 'Thriller'], 'rating': 8.8, 'box_office': 836.8}
{"title":"Inception","release_year":2010,"genres":["Action","Science Fiction","Thriller"],"rating":8.8,"box_office":836.8}


Movie(title='Inception', release_year=2010, genres=['Sci-Fi', 'Action', 'Adventure'], rating=8.8, box_office=None)

# JSON Schema

In [3]:
from langchain_openai import ChatOpenAI
from dotenv import load_dotenv
import os

load_dotenv()

llm = ChatOpenAI(
    model="deepseek-ai/deepseek-v4-pro",
    temperature=0,
    api_key=os.getenv("DEEPSEEK_API_KEY_PRO"),
    base_url="https://integrate.api.nvidia.com/v1"
)

# JSON Schema
movie_json_schema = {
    "name": "movie_schema",
    "description": "Schema for extracting movie information",
    "parameters": {
        "type": "object",
        "properties": {
            "title": {"type": "string", "description": "Name of the movie"},
            "release_year": {"type": "integer", "description": "Year the movie was released"},
            "genres": {"type": "array", "items": {"type": "string"}, "description": "List of movie genres"},
            "rating": {"type": "number", "description": "Movie rating from 1 to 10"},
            "box_office": {"type": ["number", "null"], "description": "Worldwide box office in millions USD"}
        },
        "required": ["title", "release_year", "genres", "rating", "box_office"]
    }
}

# Structured output
structured_llm = llm.with_structured_output(movie_json_schema)

result = structured_llm.invoke(
    "Give me details about the movie Inception."
)

result

{'title': 'Inception', 'release_year': 2010, 'genres': ['Action', 'Science Fiction', 'Thriller'], 'rating': 8.8, 'box_office': 836.8}


In [4]:
from typing import List, Optional
from pydantic import BaseModel, Field
from langchain_openai import ChatOpenAI
from dotenv import load_dotenv
import os

load_dotenv()

class ProductReview(BaseModel):
    product_name: str = Field(..., description="Name of the product being reviewed")
    reviewer_name: str = Field(..., description="Name of the reviewer")
    rating: float = Field(..., description="Rating from 1 to 5")
    pros: List[str] = Field(..., description="Positive aspects of the product")
    cons: List[str] = Field(..., description="Negative aspects of the product")
    review_text: str = Field(..., description="Detailed written review")
    would_recommend: bool = Field(..., description="Whether reviewer recommends product")
    purchase_date: Optional[str] = Field(None, description="Purchase date in YYYY-MM-DD format")

llm = ChatOpenAI(
    model="deepseek-ai/deepseek-v4-pro",
    temperature=0.6,
    api_key=os.getenv("DEEPSEEK_API_KEY_PRO"),
    base_url="https://integrate.api.nvidia.com/v1"
)

structured_llm = llm.with_structured_output(ProductReview)

result = structured_llm.invoke("""
I recently purchased the Bose QuietComfort 45 Noise Cancelling Headphones about three months ago, and I have been using them almost daily since then.

The noise cancellation is outstanding.
The sound quality is top-notch.
Battery life is exceptional.

However, the touch controls can be finicky,
and the case feels bulky.

Overall, I would highly recommend them.
""")

print(result)

# Dictionary format
print(result.model_dump())
print(result.model_dump_json())

product_name='Bose QuietComfort 45 Noise Cancelling Headphones' reviewer_name='Anonymous' rating=5.0 pros=['Outstanding noise cancellation', 'Top-notch sound quality', 'Exceptional battery life'] cons=['Finicky touch controls', 'Bulky case'] review_text='I recently purchased the Bose QuietComfort 45 Noise Cancelling Headphones about three months ago, and I have been using them almost daily since then. The noise cancellation is outstanding. The sound quality is top-notch. Battery life is exceptional. However, the touch controls can be finicky, and the case feels bulky. Overall, I would highly recommend them.' would_recommend=True purchase_date=None
{'product_name': 'Bose QuietComfort 45 Noise Cancelling Headphones', 'reviewer_name': 'Anonymous', 'rating': 5.0, 'pros': ['Outstanding noise cancellation', 'Top-notch sound quality', 'Exceptional battery life'], 'cons': ['Finicky touch controls', 'Bulky case'], 'review_text': 'I recently purchased the Bose QuietComfort 45 Noise Cancelling He